# Week 13: the AASIST baseline

AASIST (Jung et al., ICASSP 2022) is a widely used end-to-end anti-spoofing network.
Its published weights were trained on ASVspoof 2019 LA, the same English corpus as our
Stage-1 model. This run measures it two ways (`src/baselines/aasist.py`):

| Runs | What it answers | Time on a T4 |
|---|---|---|
| zero-shot, published weights | does an established English detector transfer to code-mixed phone-line speech? | ~10 min |
| fine-tuned on the deployed S2 training set, seeds 1234, 2025, 7, 42, 99 | same data as S2, different architecture | ~15 min each |
| fine-tuned on the extended set (248 speakers), the same seeds | same data as the overfit-fix run | ~40 min each |

Every model is scored over the phone line on: the original test pool, CM04 (Tortoise),
the RVC test half, the **60 new speakers** (CM05 XTTS and CM06 Tortoise), **CM07
IndicF5**, and the false alarms on the **language-control** genuine set. Fine-tuning
keeps the epoch with the lowest **dev** EER; no test set is used to choose anything.

**How to run.** Inputs (all three): `saikrishnareddy9/codemix-bundle-normalised`,
`saikrishnareddy9/codemix-extension-bundle` and `saikrishnareddy9/codemix-week13-clones`.
Settings: **GPU T4 x2** (one is used), Internet **On**. Then **Save Version -> Save & Run
All (Commit) -> Save**: it runs in the background, so the tab can be closed.

**Getting the results.** `/kaggle/working` keeps only `aasist_results.zip`. When the
version finishes: the notebook's page -> **Output** -> `aasist_results.zip` -> Download.
The zip is rewritten after every run, and the run stops itself at 10.5 hours.

## 1. Settings

In [ ]:
SEEDS = [1234, 2025, 7, 42, 99]
TRAIN_SETS = {  # name -> (train manifest, dev manifest); runs alternate per seed
    "s2data": ("data/manifests/codemix_adapt_train_rvc_channel20.csv",
               "data/manifests/codemix_adapt_dev_channel20.csv"),
    "extdata": ("data/manifests/codemix_adapt_train_ext_rvc_channel20.csv",
                "data/manifests/codemix_adapt_dev_channel20.csv"),
}
EPOCHS = 15
EVALS = {  # all over the phone line
    "eval_pool": "data/manifests/codemix_eval_channel20.csv",
    "cm04": "data/manifests/score_cm04_norm_channel20.csv",
    "rvc_test": "data/manifests/rvc_holdout_test_channel20.csv",
    "new_speakers": "data/manifests/score_ext_channel20.csv",
    "cm07": "data/manifests/score_cm07_channel20.csv",
    "control": "data/manifests/mucs_language_control_channel20.csv",  # genuine only
}
ZIP_NAME = "aasist_results.zip"
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH = "week13-mounika-overfit-fix"

## 2. Setup: clone, one data root, checks

In [ ]:
import glob, json, os, re, shutil, subprocess, sys, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")    # the saved output: the zip only
TEMP = Path("/kaggle/temp")
try:
    TEMP.mkdir(parents=True, exist_ok=True)
    (TEMP / ".writable").write_text("ok")
except OSError:
    TEMP = Path("/tmp/w13")
    TEMP.mkdir(parents=True, exist_ok=True)
REPO = TEMP / "codemix-deepfake-detection"
OUT = TEMP / "aasist_results"
ROOT = TEMP / "root_channel"
DEADLINE = time.time() + 10.5 * 3600
FAILURES = []


def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True,
                       cwd=REPO if REPO.exists() else None)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode and r.stderr.strip():
        print(r.stderr[-2500:])
    if check and r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r


def run_logged(args):
    """Run a long command, streaming its output, and return all of it."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    proc = subprocess.Popen(args, text=True, errors="replace", cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    with open(OUT / "run_log.txt", "a", encoding="utf-8") as log:
        log.write(f"\n$ {' '.join(args)}\n")
        for line in proc.stdout:
            lines.append(line)
            log.write(line)
            if not any(n in line for n in ("Warning", "warn(")):
                print(line, end="", flush=True)
    proc.wait()
    return proc.returncode, "".join(lines)


def save_progress():
    """Zip the results so far (never checkpoints) into /kaggle/working."""
    import zipfile
    staging = WORK / (ZIP_NAME + ".tmp")
    with zipfile.ZipFile(staging, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUT.rglob("*")):
            if f.is_file() and f.suffix != ".pt":
                zf.write(f, f.relative_to(OUT.parent))
    staging.replace(WORK / ZIP_NAME)


def tail(text, n=30):
    return "\n".join([line for line in text.splitlines() if line.strip()][-n:])


def failed(where, detail):
    FAILURES.append({"where": where, "detail": detail})
    print(f"\n!!!!!! FAILED: {where}\n{detail}\n!!!!!! recorded; the run continues\n", flush=True)
    try:
        (OUT / "FAILURES.txt").write_text(
            "\n\n".join(f"== {f['where']}\n{f['detail']}" for f in FAILURES), encoding="utf-8")
        save_progress()
    except Exception as e:
        print("(could not write FAILURES.txt:", e, ")")


def _make_writable(func, path, _):
    os.chmod(path, 0o700)
    func(path)


os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
for attempt in range(1, 4):
    if REPO.exists():
        shutil.rmtree(REPO, onerror=_make_writable)
    if subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                       f"https://{REPO_HOST}", str(REPO)]).returncode == 0:
        break
    print(f"clone attempt {attempt} failed; retrying in 20 s", flush=True)
    time.sleep(20)
else:
    raise SystemExit("Could not clone the repository after 3 attempts. Is Internet switched On?")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()
sh(f"{sys.executable} -m pip install -q soundfile==0.12.1 librosa==0.11.0", check=False)

folders = [p for p in glob.glob("/kaggle/input/**/channel/clips", recursive=True) if os.listdir(p)]
# The clones dataset holds only phone-line clips, unpacked as clips/ at its top level.
folders += [p for p in glob.glob("/kaggle/input/**/clips", recursive=True)
            if "week13-clones" in p and p not in folders and os.listdir(p)]
print("phone-line clip folders:", folders)
if len(folders) < 3:
    raise SystemExit("Attach saikrishnareddy9/codemix-bundle-normalised, "
                     "saikrishnareddy9/codemix-extension-bundle and "
                     "saikrishnareddy9/codemix-week13-clones (Add Input).")
shutil.rmtree(ROOT, ignore_errors=True)
(ROOT / "clips").mkdir(parents=True)
linked = same_name = 0
for folder in folders:
    for name in os.listdir(folder):
        target = ROOT / "clips" / name
        if target.exists():
            same_name += 1
            continue
        os.symlink(os.path.join(folder, name), target)
        linked += 1
print(f"data root {ROOT}: {linked} clips linked ({same_name} names in both datasets, first kept)")

import pandas as pd
from src.data.scoring_manifests import missing_files

manifests = set(EVALS.values()) | {m for pair in TRAIN_SETS.values() for m in pair}
for manifest in sorted(manifests):
    gone = missing_files(pd.read_csv(manifest), str(ROOT))
    print(f"  {manifest:58s} missing {len(gone)}")
    if gone:
        raise SystemExit(f"{manifest} has clips in no attached dataset, e.g. {gone[:3]} -- "
                         "are all three data inputs attached?")
print("every clip every run needs is present")
sh(f"{sys.executable} -m pytest tests/test_aasist_baseline.py tests/test_splits.py -q")

## 3. Score, fine-tune, summarise, and the download link

In [ ]:
import statistics as st
import zipfile

DEV_LINE = re.compile(r"best dev EER = ([0-9.]+)")
RUNS = []
DONE = False
FIELDS = ["eval_pool", "cm04", "rvc_test", "new_xtts", "new_tortoise", "cm07", "control_fa"]
SCORE = ["--device", "cuda", "--batch-size", "32", "--num-workers", "4"]


def score_all(name, weights):
    """Every evaluation set for one set of weights; returns the row's fields."""
    row = {}
    for ev, manifest in EVALS.items():
        tag = f"{name}__{ev}"
        code_, text = run_logged([sys.executable, "-m", "src.baselines.aasist", "score",
                                  "--weights", weights, "--manifest", manifest, "--data-root", ROOT,
                                  *SCORE, "--scores-out", OUT / f"{tag}_scores.csv",
                                  "--out", OUT / f"{tag}.json"])
        try:
            result = json.loads((OUT / f"{tag}.json").read_text()) if code_ == 0 else None
        except Exception:
            result = None
        if result is None:
            failed(f"scoring {tag} (exit {code_})", tail(text))
            continue
        if ev == "control":
            row["control_fa"] = result["summary"]["false_alarm_rate"]
        elif ev == "new_speakers":
            per = {r["attack"]: r["eer"] for r in result["per_attack"]}
            row["new_xtts"], row["new_tortoise"] = per.get("xtts_v2"), per.get("tortoise")
        else:
            row[ev] = result["pooled"]["eer"]
    return row


def summary_text():
    lines = ["STATUS: " + ("FINISHED - every run ran." if DONE else
                           f"IN PROGRESS - {len(RUNS)} run(s) finished so far."),
             "RUN HEALTH: " + ("every step ran without error." if not FAILURES else
                               f"{len(FAILURES)} step(s) FAILED -- see FAILURES.txt"),
             f"commit: {COMMIT}", "",
             "AASIST EER (%) per run, phone line; control_fa = genuine clips called fake (%)",
             f"{'run':20s}{'status':>9s}{'dev':>7s}" + "".join(f"{f:>13s}" for f in FIELDS)]
    for r in RUNS:
        dev = "-" if r.get("dev_eer") is None else f"{100 * r['dev_eer']:.2f}"
        cells = "".join(f"{'-' if r.get(f) is None else format(100 * r[f], '.2f'):>13s}" for f in FIELDS)
        lines.append(f"{r['name']:20s}{r['status']:>9s}{dev:>7s}{cells}")
    lines.append("\nMEAN +/- SD over seeds")
    for data in TRAIN_SETS:
        lines.append(f"  fine-tuned on {data}:")
        for f in FIELDS:
            v = [100 * r[f] for r in RUNS if r.get("data") == data and r.get(f) is not None]
            text = (f"{st.mean(v):6.2f} +/- {st.stdev(v):5.2f} (n={len(v)})" if len(v) >= 2
                    else ("-" if not v else f"{v[0]:6.2f} (n=1)"))
            lines.append(f"    {f:13s} {text}")
    return "\n".join(lines)


def write_summary():
    try:
        text = summary_text()
    except Exception:
        text = "The summary could not be built:\n" + traceback.format_exc()
    try:
        (OUT / "summary.txt").write_text(text, encoding="utf-8")
        (OUT / "summary.json").write_text(json.dumps({
            "mode": "full" if DONE else "in progress", "commit": COMMIT, "branch": BRANCH,
            "seeds": SEEDS, "epochs": EPOCHS, "train_sets": TRAIN_SETS, "runs": RUNS,
            "failures": FAILURES}, indent=2, default=str))
    except Exception as e:
        print("(could not write the summary:", e, ")", flush=True)
    return text


def show_download_link():
    from IPython.display import FileLink, display
    path = WORK / ZIP_NAME
    if not path.is_file():
        print("No results zip was written -- see the errors above.")
        return
    with zipfile.ZipFile(path) as zf:
        count, broken = len(zf.namelist()), zf.testzip()
    print(f"\n{ZIP_NAME}: {count} files, {path.stat().st_size / 1e6:.1f} MB,",
          "checked OK" if broken is None else f"DAMAGED at {broken}")
    os.chdir(WORK)
    print("\nCLICK THIS LINK TO DOWNLOAD THE RESULTS:")
    display(FileLink(ZIP_NAME))
    print(f"In a saved version: the notebook's page -> Output -> {ZIP_NAME} -> Download.")


def record(row, t0):
    row["minutes"] = round((time.time() - t0) / 60, 1)
    if row["status"] != "failed":
        row["status"] = "ok" if all(row.get(f) is not None for f in FIELDS) else "partial"
    RUNS.append(row)
    (OUT / "runs.json").write_text(json.dumps(RUNS, indent=2))
    write_summary()
    save_progress()
    shown = "  ".join(f"{f} {100 * row[f]:.2f}" for f in FIELDS if row.get(f) is not None)
    print(f"\n[{row['name']}] {row['status']}  {shown}  ({row['minutes']} min)\n", flush=True)


def zero_shot():
    t0 = time.time()
    row = {"name": "zero_shot", "data": None, "seed": None, "dev_eer": None, "status": "running"}
    try:
        row.update(score_all("zero_shot", "third_party/aasist/AASIST.pth"))
    except Exception:
        row["status"] = "failed"
        failed("zero_shot: unexpected error", traceback.format_exc())
    record(row, t0)


def fine_tune(data, seed):
    name = f"{data}_s{seed}"
    t0 = time.time()
    row = {"name": name, "data": data, "seed": seed, "dev_eer": None, "status": "running"}
    out_dir = TEMP / "aasist_ckpt" / name
    try:
        train_m, dev_m = TRAIN_SETS[data]
        code_, text = run_logged([sys.executable, "-m", "src.baselines.aasist", "train",
                                  "--train-manifest", train_m, "--dev-manifest", dev_m,
                                  "--data-root", ROOT, "--out-dir", out_dir, "--epochs", EPOCHS,
                                  "--seed", seed, "--device", "cuda", "--num-workers", "4"])
        ckpt = out_dir / "best.pt"
        if code_ or not ckpt.is_file():
            row["status"] = "failed"
            failed(f"training {name} (exit {code_})", tail(text))
        else:
            dev = DEV_LINE.findall(text)
            row["dev_eer"] = float(dev[-1]) if dev else None
            row.update(score_all(name, ckpt))
    except Exception:
        row["status"] = "failed"
        failed(f"{name}: unexpected error", traceback.format_exc())
    finally:
        shutil.rmtree(out_dir, ignore_errors=True)
    record(row, t0)


try:
    zero_shot()
    jobs = [(data, seed) for seed in SEEDS for data in TRAIN_SETS]
    for data, seed in jobs:
        if time.time() > DEADLINE:
            failed("time limit", f"stopped before {data}_s{seed}; finished runs are kept")
            break
        fine_tune(data, seed)
    else:
        DONE = True
        print("\nALL RUNS FINISHED\n", flush=True)
finally:
    print(write_summary(), flush=True)
    save_progress()
    show_download_link()

## 4. Optional: the download link again

Section 3 already ends with the link. Run this only if you need it again.

In [ ]:
import os, zipfile
from pathlib import Path
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
if Path(ZIP_NAME).is_file():
    with zipfile.ZipFile(ZIP_NAME) as zf:
        print(f"{ZIP_NAME}: {len(zf.namelist())} files, checked", "OK" if zf.testzip() is None else "DAMAGED")
    display(FileLink(ZIP_NAME))
else:
    print("No results zip yet -- section 3 writes it after its first finished run.")